# UC2b - Time-resolved kidney fibrosis as one AnnNet object

UC2b is the time-resolved sibling of UC2. It builds the *same* heterogeneous map of
human cell biology - OmniPath signaling, DoRothEA regulation, OmniPath complexes,
and the Human-GEM metabolic model - but replaces UC2's second Kivela aspect. Where
UC2's second aspect was *structural* (`complex` = member / monomer), UC2b's is
*temporal*: `time`, driven by a real time course.

The data is the Saez-lab kidney-fibrosis multi-omics study: human kidney PDGFRb+
mesenchymal cells stimulated with TGF-beta, profiled by transcriptomics,
proteomics, phosphoproteomics, and secretomics across seven timepoints. Both
aspects now carry biological meaning.

- `mechanism` in {signaling, regulatory, metabolic}: the process an edge belongs to.
- `time` in {0h, 1h, 12h, 24h, 48h, 72h, 96h}: when an entity is *responsive*.

A node-layer is therefore e.g. `(prot:COL1A1, signaling, 24h)`. This is a software
case study and reports no biological finding.

## The responsive gate

UC2 gated its universe on a generic protein-atlas evidence flag - a near-null
filter. UC2b gates on *measured response in this cell system*: a gene symbol enters
a time layer only where the study calls it differential at that timepoint
(adj.P < 0.05, gene-symbol union across all four modalities). No fold-change floor,
so the early layers stay populated. The cell below reads the study's differential
results and caches, per timepoint, the responsive symbols (with the strongest
logFC and its `strong` = |logFC|>1 flag) and the measured universe. Later notebooks
reload these parquet caches, so no R is needed at build time.

In [1]:
import pyreadr
import polars as pl

import uc2b_common as uc

diff = pyreadr.read_r(str(uc.DIFF_RDATA))["diff_results"]
d = (pl.from_pandas(diff[["modality", "feature_id", "logFC", "adj.P.Val", "time"]])
     .rename({"adj.P.Val": "adjp"})
     .filter(pl.col("feature_id").is_not_null())
     .with_columns(symbol=pl.col("feature_id").map_elements(uc.symbol_of, return_dtype=pl.Utf8)))

# Measured universe: every symbol the study quantified, per modality (the context gate).
measured = d.select("symbol", "modality").unique()

# Responsive: adj.P < threshold, aggregated from site/feature rows up to (symbol, time),
# keeping the strongest-magnitude logFC as the node attribute.
responsive = (d.filter(pl.col("adjp") < uc.ADJP_THRESHOLD)
              .with_columns(abslfc=pl.col("logFC").abs())
              .sort("abslfc", descending=True)
              .group_by("symbol", "time")
              .agg(best_logFC=pl.col("logFC").first(),
                   min_adjp=pl.col("adjp").min(),
                   n_modalities=pl.col("modality").n_unique())
              .with_columns(strong=pl.col("best_logFC").abs() > uc.STRONG_LOGFC))

responsive.write_parquet(str(uc.RESPONSIVE))
measured.write_parquet(str(uc.MEASURED))

counts = (responsive.group_by("time")
          .agg(responsive=pl.len(), strong=pl.col("strong").sum())
          .to_pandas().set_index("time").reindex(["0.08h", *uc.TIMES]))
print(f"measured universe: {measured['symbol'].n_unique():,} symbols")
print(f"responsive (symbol, time) rows: {responsive.height:,}\n")
print(counts.to_string())

measured universe: 14,966 symbols
responsive (symbol, time) rows: 23,271

       responsive  strong
time                     
0.08h          12       2
1h            529      96
12h          2762     489
24h          3297     602
48h          4464     716
72h          5078     791
96h          7129    1014
